## Установка

In [2]:
from sklearn.metrics import accuracy_score
from sklearn.utils.extmath import softmax
!pip3 install torch torchvision


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Проверка установки

In [3]:
import torch
import torch.nn.functional as F
from torch.autograd import grad
import torch.nn as nn

In [4]:
x = torch.rand(5, 3)
print(x)

tensor([[0.6116, 0.5915, 0.1090],
        [0.5692, 0.4613, 0.2763],
        [0.3809, 0.6416, 0.7853],
        [0.5523, 0.6857, 0.7595],
        [0.0588, 0.0203, 0.1801]])


## Проверка работы с видюхой

In [5]:
    if torch.backends.mps.is_available():
        device = torch.device("mps")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")

    print(device)

cpu


## Знакомство с тензорами

In [6]:
tensor0d = torch.tensor(1)
print(tensor0d)

tensor1d = torch.tensor([1,2,3])
print(tensor1d)

tensor2d = torch.tensor([[1,2],
                         [3,4]])
print(tensor2d)


tensor3d = torch.tensor([[[1,2],[3,4]],
                        [[5,6],[7,8]]])
print(tensor3d)

tensor(1)
tensor([1, 2, 3])
tensor([[1, 2],
        [3, 4]])
tensor([[[1, 2],
         [3, 4]],

        [[5, 6],
         [7, 8]]])


In [7]:
print(tensor1d.dtype)
print(tensor3d.shape)

x = tensor3d.reshape(2,4)
print(x.shape)
print(x)

y = tensor3d.view(2,4)
print(y.shape)
print(y)

torch.int64
torch.Size([2, 2, 2])
torch.Size([2, 4])
tensor([[1, 2, 3, 4],
        [5, 6, 7, 8]])
torch.Size([2, 4])
tensor([[1, 2, 3, 4],
        [5, 6, 7, 8]])


In [8]:
print(tensor2d.T) # транспонирование

print(tensor2d.matmul(tensor2d.T)) # умножение
print(tensor2d @ tensor2d.T) # умножение

tensor([[1, 3],
        [2, 4]])
tensor([[ 5, 11],
        [11, 25]])
tensor([[ 5, 11],
        [11, 25]])


In [9]:
# y = w*x+b

y = torch.tensor([1.0])
x = torch.tensor([1.1])
w = torch.tensor([2.2], requires_grad=True) # веса; для вычисления производной
b = torch.tensor([0.0], requires_grad=True)

z = w*x+b # типа перпцетрон с подбором множителя и отклонением

a=torch.sigmoid(z) # добавляем нелинейность через активационную функцию

loss = F.binary_cross_entropy(a,y) # потери

grad_L_w = grad(loss,w, retain_graph=True)
grad_L_b = grad(loss,b, retain_graph=True)

print(grad_L_w, grad_L_b)

## другая форма записи

loss.backward()
print(w.grad, b.grad)

(tensor([-0.0898]),) (tensor([-0.0817]),)
tensor([-0.0898]) tensor([-0.0817])


## Сделаем нейросеть прямого распространения

In [47]:
class NeuralNetwork(nn.Module):
    # конструктор
    def __init__(self, num_inputs, num_outputs): # кол-во входных и выходных нейронов
        super().__init__() # вызываем конструтктор у папы

        self.layers = nn.Sequential(
            # 1st layer
            nn.Linear(num_inputs, 10), # размерность слоя; как в примере выше перемножаем нейроны
            nn.ReLU(), # активационная функция

            # 2nd layer
            nn.Linear(10, 10),
            nn.ReLU(),

            # 3rd layer
            nn.Linear(10, num_outputs),
            nn.Sigmoid(), #активационная функция, которая перводит значения в значения от 0 до 1
        )

    # функция, обеспечивающая прямой проход
    def forward(self, x):
        return self.layers(x) # прогоняем вектор x через все слои


In [33]:
model = NeuralNetwork(4,2)
print(model)

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=4, out_features=10, bias=True)
    (1): ReLU()
    (2): Linear(in_features=10, out_features=10, bias=True)
    (3): ReLU()
    (4): Linear(in_features=10, out_features=2, bias=True)
  )
)


In [12]:
print(model.layers)

Sequential(
  (0): Linear(in_features=4, out_features=10, bias=True)
  (1): ReLU()
  (2): Linear(in_features=10, out_features=10, bias=True)
  (3): ReLU()
  (4): Linear(in_features=10, out_features=2, bias=True)
)


In [36]:
print(model.layers[0].weight)

Parameter containing:
tensor([[-0.2636, -0.2734,  0.3005, -0.3308],
        [-0.2350,  0.2720, -0.3718,  0.2452],
        [ 0.3045,  0.1357,  0.0896,  0.1933],
        [ 0.3782,  0.0407, -0.3600,  0.4613],
        [ 0.3666, -0.0116, -0.2923, -0.1937],
        [-0.4415,  0.3314, -0.0434,  0.3445],
        [ 0.1883, -0.0992, -0.3197,  0.1740],
        [-0.3208, -0.1111,  0.2972, -0.2723],
        [-0.0245, -0.0679,  0.0573,  0.4814],
        [ 0.2923, -0.1877,  0.4425, -0.1626]], requires_grad=True)


In [15]:
params = [p for p in model.parameters() if p.requires_grad]
params

[Parameter containing:
 tensor([[-0.2656,  0.4249, -0.0842, -0.2318],
         [-0.3183, -0.3540,  0.2550, -0.4418],
         [-0.0356,  0.0397, -0.4097,  0.3027],
         [-0.2001, -0.4763, -0.0141,  0.1691],
         [ 0.4301, -0.2851,  0.2389,  0.4700],
         [ 0.3961, -0.0125,  0.3847,  0.3166],
         [-0.2029, -0.0928, -0.3977,  0.0822],
         [-0.2603, -0.3286,  0.3168, -0.3874],
         [-0.4502, -0.1466, -0.0686,  0.1909],
         [ 0.3393,  0.0400,  0.3155,  0.0616]], requires_grad=True),
 Parameter containing:
 tensor([-0.2834, -0.3707,  0.2902,  0.1915,  0.3268,  0.0176, -0.1887, -0.3529,
         -0.2085, -0.4085], requires_grad=True),
 Parameter containing:
 tensor([[ 0.1457,  0.1962, -0.2054, -0.0694, -0.1064,  0.1992,  0.0569, -0.3038,
           0.2234, -0.0338],
         [-0.2581, -0.0975,  0.0217,  0.2880, -0.2998,  0.2611, -0.2350, -0.2442,
           0.0307, -0.0906],
         [-0.1099, -0.1465, -0.0417,  0.2444, -0.2944, -0.0461,  0.2422, -0.0381,
     

In [18]:
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
num_params

182

In [42]:
torch.manual_seed(42) # сид для обучения модели - веса будут одинаковые
model_1 = NeuralNetwork(4,2)
print(model_1.layers[0].weight)

Parameter containing:
tensor([[ 0.3823,  0.4150, -0.1171,  0.4593],
        [-0.1096,  0.1009, -0.2434,  0.2936],
        [ 0.4408, -0.3668,  0.4346,  0.0936],
        [ 0.3694,  0.0677,  0.2411, -0.0706],
        [ 0.3854,  0.0739, -0.2334,  0.1274],
        [-0.2304, -0.0586, -0.2031,  0.3317],
        [-0.3947, -0.2305, -0.1412, -0.3006],
        [ 0.0472, -0.4938,  0.4516, -0.4247],
        [ 0.3860,  0.0832, -0.1624,  0.3090],
        [ 0.0779,  0.4040,  0.0547, -0.1577]], requires_grad=True)


In [45]:
x = torch.rand((1,4))
out = model(x)

print(out)

tensor([[ 0.0194, -0.0300]], grad_fn=<AddmmBackward0>)


In [50]:
with torch.no_grad(): # декоратор чтобы не высчитывать градиент
    x = torch.rand((1,4))
    out = model(x)

print(out)

tensor([[-0.0167, -0.0552]])


In [59]:
model_2 = NeuralNetwork(4,3)

with torch.no_grad(): # декоратор чтобы не высчитывать градиент
    x = torch.rand((1,4))
    y = model_2(x)

    out = torch.softmax(y,dim=1) # активиционная функция - нормализация до суммы элементов = 1

print(out)

tensor([[0.3153, 0.3355, 0.3492]])


In [62]:
X_train = torch.tensor([
    [-1.0,3.4],
    [-1.0,2.4],
    [-5.5,1.1],
    [2.0,-1.4],
    [4.0,-2.4],
])

y_train = [0,0,0,1,1]

X_test = torch.tensor([
    [-0.3,2.5],
    [1.0,-2.0],
])

y_test = [0,1]

In [60]:
class MyDataset(torch.utils.data.Dataset): # ДатаЛоудер
    def __init__(self,X,y):
        self.features = X
        self.labels = y

    def __getitem__(self, index):
        x = self.features[index]
        y = self.labels[index]
        return x,y

    def __len__(self):
        return len(self.features)

In [64]:
train_dataset = MyDataset(X_train,y_train)
test_dataset = MyDataset(X_test,y_test)

In [65]:
print(len(train_dataset))
print(len(test_dataset))

5
2


In [67]:
train_dataloader = torch.utils.data.DataLoader(
    train_dataset,batch_size=2,shuffle=True, num_workers=0)

test_dataloader = torch.utils.data.DataLoader(
    test_dataset,batch_size=2,shuffle=False, num_workers=0)

for(idx, (x,y)) in enumerate(train_dataloader):
    print(f'Batch #{idx}, \nx:{x}, \ny:{y}')

Batch #0, 
x:tensor([[ 4.0000, -2.4000],
        [-5.5000,  1.1000]]), 
y:tensor([1, 0])
Batch #1, 
x:tensor([[-1.0000,  2.4000],
        [-1.0000,  3.4000]]), 
y:tensor([0, 0])
Batch #2, 
x:tensor([[ 2.0000, -1.4000]]), 
y:tensor([1])


In [69]:
model_3 = NeuralNetwork(num_inputs=2,num_outputs=2)

optimizer = torch.optim.SGD(model_3.parameters(), lr=0.01) # оптимайзер поиска градиента

num_epochs = 10
for epoch in range(num_epochs):
    model_3.train()

    for (idx,(x,y)) in enumerate(train_dataloader):
        model_3_result = model_3(x)

        loss = F.cross_entropy(model_3_result,y)

        optimizer.zero_grad() # сбрасываем градиент
        loss.backward() # считаем его
        optimizer.step() # обновляем параметр модели

        print(f'Batch: #{epoch}/{idx}, loss: {loss:.2f}')

Batch: #0/0, loss: 0.69
Batch: #0/1, loss: 0.68
Batch: #0/2, loss: 0.66
Batch: #1/0, loss: 0.69
Batch: #1/1, loss: 0.66
Batch: #1/2, loss: 0.71
Batch: #2/0, loss: 0.69
Batch: #2/1, loss: 0.69
Batch: #2/2, loss: 0.66
Batch: #3/0, loss: 0.69
Batch: #3/1, loss: 0.69
Batch: #3/2, loss: 0.65
Batch: #4/0, loss: 0.66
Batch: #4/1, loss: 0.72
Batch: #4/2, loss: 0.65
Batch: #5/0, loss: 0.72
Batch: #5/1, loss: 0.66
Batch: #5/2, loss: 0.65
Batch: #6/0, loss: 0.69
Batch: #6/1, loss: 0.68
Batch: #6/2, loss: 0.65
Batch: #7/0, loss: 0.65
Batch: #7/1, loss: 0.68
Batch: #7/2, loss: 0.72
Batch: #8/0, loss: 0.69
Batch: #8/1, loss: 0.65
Batch: #8/2, loss: 0.72
Batch: #9/0, loss: 0.68
Batch: #9/1, loss: 0.68
Batch: #9/2, loss: 0.66


In [73]:
model_3.eval() # переводим модель в режим подсяёта эффективности

for (idx,(x,y)) in enumerate(test_dataloader):
    with torch.no_grad():
        outputs = torch.argmax(torch.softmax(model_3(x), dim=1), dim =1)
        print(f'Batch: #{idx}, output: {outputs}, y: {y}')

Batch: #0, output: tensor([0, 0]), y: tensor([0, 1])


In [ ]:
tensor0d.to(device) # переносим на вычисляемый модуль(gpu cpu)

## ДЗ

Iris -> NN -> accuracy